In [1]:
import numpy as np
from pathlib import Path

from functions import *

from minerva.transforms.transform import *
from minerva.transforms.random_transform import *

from minerva.data.readers import TiffReader, PNGReader
from minerva.data.datasets import SimpleDataset
from minerva.data.data_modules import MinervaDataModule

from minerva.models.ssl.byol import BYOL
from minerva.models.nets.image.deeplabv3 import DeepLabV3Backbone

from minerva.pipelines.lightning_pipeline import SimpleLightningPipeline
from lightning.pytorch.loggers.csv_logs import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning import Trainer

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data

## Variables

In [ ]:
repetition = 0
pretrain_data = 'sup'
finetune_data = 'seam_ai'
cap = 0.01
data_path = '/workspaces/shared_data/seam_ai_datasets/seam_ai'

In [ ]:
save_name = (
    f"V{repetition}_pre_{pretrain_data}_train_{finetune_data}_cap_{cap*100:.2f}%"
)

# Transforms

In [4]:
if finetune_data == "f3" or finetune_data == "f3_N":
    padding = Padding(256, 704)
elif finetune_data == "seam_ai" or finetune_data == "seam_ai_N":
    padding = Padding(1008, 592)

# Image Visualization 

In [5]:
# transpose = Transpose((1, 2, 0))

# image_list = [
#     transpose(padded_image),
#     transpose(padded_label)
# ]
# plot_images(image_list)

# Dataset

In [6]:
image_path = f"{data_path}/images"
label_path = f"{data_path}/annotations"

train_data_reader = TiffReader(path=f'{image_path}/train')
train_label_reader = PNGReader(path=f"{label_path}/train")

val_data_reader = TiffReader(path=f'{image_path}/val')
val_label_reader = PNGReader(path=f"{label_path}/val")


train_dataset = SimpleDataset(
    readers=[
        train_data_reader,
        train_label_reader,
    ],
    transforms=padding,
    return_single=False,
)

val_dataset = SimpleDataset(
    readers=[
        val_data_reader,
        val_label_reader,
    ],
    transforms=padding,
    return_single=False
)

# DataModule

In [7]:
batch_size = 8
seed = 42

In [8]:
data_module = CapDataModule(
    cap_train=cap,
    cap_val=1,
    cap_test=1,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=batch_size,
    drop_last=True,
    shuffle_train=True
)

In [9]:
# if cap == 1:
#     data_module = MinervaDataModule(
#         train_dataset=train_dataset,
#         batch_size=batch_size,
#         drop_last=True,
#         shuffle_train=True,
#     )
# else:
#     raise NotImplementedError(
#         f'Missing diferent caps data module!'
#     )

# Model

In [10]:
learning_rate = 0.001
freeze = False
import_root_path = None

In [11]:
model = get_model(
    pretrain_data,
    learning_rate,
    freeze,
    repetition,
    import_root_path,
)

2025-04-22 17:04:10 - minerva - INFO - No model loaded!


# Trainer

In [12]:
ckpt_path = f'ckpt/pretrain/{repetition}'
logs_path = f'logs/pretrain/{repetition}'         
gpus = [0]     
num_epochs = 10 

In [13]:
log_dir = Path(logs_path) / save_name / finetune_data
ckpt_dir = Path(ckpt_path) / save_name / finetune_data
logger = CSVLogger(log_dir, name=save_name, version=finetune_data)
ckpt_callback = ModelCheckpoint(save_top_k=1, 
                                save_last=True, 
                                dirpath=ckpt_dir,
                                monitor="val_IoU",
                                mode="max")

In [14]:
trainer = Trainer(
    accelerator="gpu",
    logger=logger,
    callbacks=[ckpt_callback],
    max_epochs=num_epochs,
    strategy="auto",
    devices=gpus,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


# Pipeline

In [15]:
pipeline = SimpleLightningPipeline(
    model=model,
    trainer=trainer,
    log_dir=log_dir,
    save_run_status=True,
)


In [16]:
pipeline.run(data_module, task="fit")


/home/vscode/.local/lib/python3.10/site-packages/lightning/fabric/utilities/seed.py:42: No seed found, seed set to 0
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name     | Type                    | Params | Mode 
-------------------------------------------------------------
0 | backbone | DeepLabV3Backbone       | 25.6 M | train
1 | fc       | DeepLabV3PredictionHead | 16.1 M | train
2 | loss_fn  | CrossEntropyLoss        | 0      | train
-------------------------------------------------------------
41.7 M    Trainable params
0         Non-trainable params
41.7 M    Total params
166.736   Total estimated model params size (MB)
186       Modules in train mode
0         Modules in eval mode


** Seed set to: 0 **
Pipeline info saved at: /workspaces/Seismic-Byol/dev-seismic-byol/logs/pretrain/0/V0_pre_sup_train_seam_ai_cap_10%/seam_ai/run_2025-04-22-17-04-110cbe3d59.yaml
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/vscode/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/home/vscode/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/vscode/.local/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (14) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0:  50%|█████     | 7/14 [00:03<00:03,  1.92it/s, v_num=m_ai]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
!nvidia-smi

Tue Apr 22 16:32:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:05:00.0 Off |                  Off |
| 30%   42C    P8             25W /  450W |      31MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----